# GPTQ visual walkthrough — **3×3 weight matrices**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_1A.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_1A.ipynb

**Run all cells top → bottom.** Every step shows the actual numbers as heatmaps.

```
Setup → classes → see W (3×3) → see X (3×3) → RTN visual → H = XᵀX visual → GPTQ column-by-column → scorecard
```

**Toy network** (each weight is **3×3**):
```
x (3) → fc1 [W₁ 3×3] → ReLU → fc2 [W₂ 3×3] → ReLU → fc3 [W₃ 3×3] → y (3)
```

**PyTorch note:** `nn.Linear(3,3)` stores `weight` as `[out, in]` = **3 rows × 3 cols**.

## Setup — imports, config & visual helpers

**Why required:**
- **PyTorch** — matrix math
- **matplotlib** — heatmaps so you *see* $W$, $X$, $H$, $Q$ at every step
- **`show_matrix()`** — one helper used everywhere for consistent plots

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "matplotlib"])

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPTQ_DAMPING = 0.01
QMAX = 7  # int4 symmetric: 2^(4-1)-1


def show_matrix(M, title, annotate=True, figsize=(3.8, 3.2)):
    """Heatmap + number labels for a small matrix."""
    A = M.detach().cpu().float().numpy()
    vmax = max(np.abs(A).max(), 1e-6)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(A, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(A.shape[1]))
    ax.set_yticks(range(A.shape[0]))
    ax.set_xticklabels([f"c{j}" for j in range(A.shape[1])])
    ax.set_yticklabels([f"r{i}" for i in range(A.shape[0])])
    if annotate:
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=11,
                        color="white" if abs(A[i,j]) > 0.55 * vmax else "black")
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


def show_row(matrices, titles, suptitle=""):
    """Side-by-side heatmaps (e.g. W vs W_hat vs error)."""
    n = len(matrices)
    fig, axes = plt.subplots(1, n, figsize=(3.8 * n, 3.2))
    if n == 1:
        axes = [axes]
    vmax = max(max(np.abs(m.detach().cpu().float().numpy()).max(), 1e-6) for m in matrices)
    for ax, M, t in zip(axes, matrices, titles):
        A = M.detach().cpu().float().numpy()
        im = ax.imshow(A, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_xticks(range(A.shape[1]))
        ax.set_yticks(range(A.shape[0]))
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if abs(A[i,j]) > 0.55 * vmax else "black")
        ax.set_title(t, fontsize=10)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


def show_weight_activation(W, X, suptitle=""):
    """Left = weight W, Right = activation X, then output y = X @ Wᵀ."""
    show_row(
        [W, X],
        ["Weight W (left)\n3×3  [out × in]", "Activation X (right)\n3×3  [batch × in]"],
        suptitle=suptitle or "Linear layer — weight (left) · activation (right)",
    )
    with torch.no_grad():
        y = F.linear(X.float(), W.float())
    show_matrix(y, "Output y = X @ Wᵀ  (3×3)")

print(f"Device: {DEVICE}")

## `SymmetricQuantizer` — int4 pack / unpack

**Why required:** same int4 formula for RTN and GPTQ.

| Step | Formula |
|------|--------|
| scale | $s_j = \max_k \|W_{jk}\| / 7$ |
| quant | $Q_{jk} = \mathrm{round}(W_{jk}/s_j)$ |
| dequant | $\hat W_{jk} = Q_{jk} \cdot s_j$ |

In [ ]:
class SymmetricQuantizer:
    def __init__(self, n_bits=4):
        self.qmax = 2 ** (n_bits - 1) - 1

    def quantize(self, W):
        s = W.abs().amax(1).clamp(min=1e-8) / self.qmax
        q = torch.round(W / s.unsqueeze(1)).clamp(-self.qmax - 1, self.qmax).to(torch.int8)
        return q, s

    def dequantize(self, q, s):
        return q.float() * s.unsqueeze(1)

print("SymmetricQuantizer OK")

### VISUAL — int4 on a single 3×3 weight

Watch each **row** get its own scale $s_j$, then integers $Q$ map back to $\hat W$.

In [ ]:
W_demo = torch.tensor([
    [ 0.50, -0.30,  0.20],
    [ 0.10,  0.80, -0.40],
    [-0.25,  0.15,  0.60],
])
sq = SymmetricQuantizer()
Q_demo, s_demo = sq.quantize(W_demo)
W_hat_demo = sq.dequantize(Q_demo, s_demo)

print("Per-row scales s (divide max|row| by 7):")
for j, sv in enumerate(s_demo):
    print(f"  row {j}: s={sv:.4f}  (max|W[{j}]|={W_demo[j].abs().max():.2f})")

show_row([W_demo, W_hat_demo, W_demo - W_hat_demo],
         ["fp16  W", "dequant Ŵ", "error W−Ŵ"],
         suptitle="Step: symmetric int4 (RTN building block)")
print("Integer codes Q:\n", Q_demo.numpy())

## `QuantState` — packed weight bag

**Why required:** pass `(Q, scales, bias)` from quantizer → `QuantLinear`.

In [ ]:
@dataclass
class QuantState:
    q: torch.Tensor
    s: torch.Tensor
    bias: torch.Tensor | None
    gptq: bool = False

print("QuantState OK")

## `QuantLinear` — int4 layer, fp forward

**Why required:** swap into model; forward dequantizes $\hat W$ then computes $y = X W^\top$.

**Layout (every linear layer):**

```
┌─────────────────┐     ┌─────────────────┐
│  Weight W       │     │  Activation X   │
│  (LEFT) 3×3     │  ×  │  (RIGHT) 3×3    │  →  y = X @ Wᵀ
│  [out × in]     │     │  [batch × in]   │
└─────────────────┘     └─────────────────┘
```

- **Left** = learned weights $W$ (what we quantize to int4)
- **Right** = inputs $X$ entering the layer (what hooks capture as $X_l$)

In [ ]:
class QuantLinear(nn.Module):
    def __init__(self, in_f, out_f, state: QuantState):
        super().__init__()
        self.register_buffer("q", state.q)
        self.register_buffer("s", state.s)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None
        self._sq = SymmetricQuantizer()

    def forward(self, x):
        W = self._sq.dequantize(self.q, self.s).to(x.dtype)  # left: weight
        return F.linear(x, W, self.bias)  # right: activation x  →  y = x @ Wᵀ

print("QuantLinear OK")

# VISUAL — weight (left) + activation (right) → output
W_vis = torch.tensor([
    [ 0.50, -0.30,  0.20],
    [ 0.10,  0.80, -0.40],
    [-0.25,  0.15,  0.60],
])
X_vis = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.3],
    [0.5, 0.5, 0.0],
])
show_weight_activation(W_vis, X_vis, suptitle="QuantLinear math: dequant Ŵ (left) uses same shape as W")

## `rtn_quantize()` — naive round-to-nearest

**Why required:** baseline — no calibration, just round $W$.

In [ ]:
def rtn_quantize(layer: nn.Linear) -> QuantState:
    sq = SymmetricQuantizer()
    q, s = sq.quantize(layer.weight.data.float())
    b = layer.bias.data.clone() if layer.bias is not None else None
    return QuantState(q, s, b, gptq=False)

print("rtn_quantize OK")

## `gptq_quantize()` — Hessian-aware (column-by-column)

**Why required:** uses calibration $X$ → $H=X^\top X$ → fix error on unquantized columns.

In [ ]:
def gptq_quantize(layer: nn.Linear, X, damping=GPTQ_DAMPING, verbose=False):
    """Returns (QuantState, snapshots) if verbose else QuantState only."""
    sq = SymmetricQuantizer()
    if X.dim() == 3:
        X = X.reshape(-1, X.shape[-1])
    X = X.float()
    H = X.t() @ X

    W = layer.weight.data.float().clone()
    W0 = W.clone()
    dead = torch.diag(H) == 0
    H[dead, dead] = 1.0
    W[:, dead] = 0.0
    idx = torch.arange(H.shape[0], device=H.device)
    H[idx, idx] += damping * H.diag().mean()

    L = torch.linalg.cholesky(H)
    Hinv = torch.linalg.cholesky(torch.linalg.cholesky_inverse(L), upper=True)

    Q = torch.zeros_like(W)
    snapshots = [("start W", W.clone())]
    for col in range(W.shape[1]):
        w = W[:, col]
        sc = w.abs().max().clamp(min=1e-8) / sq.qmax
        q = torch.round(w / sc).clamp(-sq.qmax - 1, sq.qmax)
        Q[:, col] = q
        err = (w - q * sc) / Hinv[col, col]
        W[:, col:] -= err.unsqueeze(1) @ Hinv[col, col:].unsqueeze(0)
        if verbose:
            snapshots.append((f"after col {col}", W.clone()))

    q, s = sq.quantize(Q)
    b = layer.bias.data.clone() if layer.bias is not None else None
    state = QuantState(q, s, b, gptq=True)
    if verbose:
        return state, W0, H, Hinv, Q, snapshots
    return state

print("gptq_quantize OK")

## `replace_layer()` & `output_mse()` — helpers

**Why required:**
- `replace_layer` — swap `nn.Linear` → `QuantLinear` by name
- `output_mse` — score **output** drift (what matters), not weight drift

In [ ]:
def replace_layer(model, name, new_mod):
    parent, _, child = name.rpartition(".")
    p = model.get_submodule(parent) if parent else model
    setattr(p, child, new_mod)


def output_mse(fp_layer, quant_layer, X):
    with torch.no_grad():
        y0 = fp_layer(X) if not isinstance(fp_layer, nn.Linear) else F.linear(X, fp_layer.weight, fp_layer.bias)
        y1 = quant_layer(X)
    return (y0 - y1).pow(2).mean().item()

print("helpers OK")

## `Toy3Layer` — 3 layers, each **W is 3×3**

**Why required:** small enough to print every number; deep enough to see error compound.

We use **fixed** weights (not random) so heatmaps stay readable.

In [ ]:
class Toy3Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 3, bias=False)
        self.fc2 = nn.Linear(3, 3, bias=False)
        self.fc3 = nn.Linear(3, 3, bias=False)
        # Hand-picked 3×3 weights (easy to read on heatmaps)
        self.fc1.weight.data = torch.tensor([
            [ 0.50, -0.30,  0.20],
            [ 0.10,  0.80, -0.40],
            [-0.25,  0.15,  0.60],
        ])
        self.fc2.weight.data = torch.tensor([
            [ 0.40,  0.20, -0.10],
            [-0.20,  0.70,  0.30],
            [ 0.15, -0.35,  0.55],
        ])
        self.fc3.weight.data = torch.tensor([
            [ 0.60, -0.10,  0.25],
            [ 0.05,  0.45, -0.30],
            [-0.20,  0.35,  0.50],
        ])

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        return self.fc3(h2)


# Calibration X: 3 samples × 3 features (displayed as 3×3)
X = torch.tensor([
    [1.0, 0.0, 0.5],
    [0.0, 1.0, 0.3],
    [0.5, 0.5, 0.0],
], device=DEVICE)

ref = Toy3Layer().to(DEVICE).eval()
with torch.no_grad():
    y_ref = ref(X)

print("Toy3Layer OK")

### VISUAL — Step 1: all three 3×3 weight matrices $W_1, W_2, W_3$

In [ ]:
show_row([ref.fc1.weight.data, ref.fc2.weight.data, ref.fc3.weight.data],
         ["W₁  fc1 (3×3)", "W₂  fc2 (3×3)", "W₃  fc3 (3×3)"],
         suptitle="Step 1 — fp16 weights (red=+, blue=−)")

### VISUAL — Step 2: forward pass — **weight (left)** · **activation (right)**

Each layer: multiply **activation** on the right by **weight** on the left.

| Layer | Left (weight 3×3) | Right (activation 3×3) | Formula |
|-------|-------------------|------------------------|---------|
| fc1 | $W_1$ | $X$ (input) | $h_1 = \mathrm{ReLU}(X W_1^\top)$ |
| fc2 | $W_2$ | $h_1$ | $h_2 = \mathrm{ReLU}(h_1 W_2^\top)$ |
| fc3 | $W_3$ | $h_2$ | $y = h_2 W_3^\top$ |

In [ ]:
def show_weight_activation(W, A, layer_name, formula):
    """Left = weight matrix, Right = activation entering this layer."""
    show_row(
        [W, A],
        [f"{layer_name}  W  (weight)", f"{layer_name}  activation"],
        suptitle=f"{formula}   ←  activation @ Wᵀ",
    )


with torch.no_grad():
    h1 = F.relu(ref.fc1(X))
    h2 = F.relu(ref.fc2(h1))

# fc1: weight W₁ (left) · input X (right)
show_weight_activation(
    ref.fc1.weight.data, X,
    "fc1", "h₁ = ReLU(X · W₁ᵀ)",
)

# fc2: weight W₂ (left) · activation h₁ (right)
show_weight_activation(
    ref.fc2.weight.data, h1,
    "fc2", "h₂ = ReLU(h₁ · W₂ᵀ)",
)

# fc3: weight W₃ (left) · activation h₂ (right)
show_weight_activation(
    ref.fc3.weight.data, h2,
    "fc3", "y = h₂ · W₃ᵀ",
)

show_matrix(y_ref, "final output  y  (3 samples × 3 outputs)")

## `LayerInputRecorder` — hook class

PyTorch calls `recorder(...)` **during `model(X)`**, not when you register the hook.

| When | What happens | `captured` dict |
|------|----------------|-----------------|
| **Setup** | `register_forward_hook(recorder)` | still `{}` |
| **`model(X)`** | `recorder.__call__` runs after each `fc` | fills `fc1`, `fc2`, `fc3` |
| **Cleanup** | `handle.remove()` | unchanged |

```
recorder = LayerInputRecorder("fc1", captured)
layer.register_forward_hook(recorder)   # install camera — no recording yet
model(X)                                # camera records X entering each layer
```


In [ ]:
class LayerInputRecorder:
    def __init__(self, layer_name, storage, verbose=False):
        self.layer_name = layer_name
        self.storage = storage
        self.verbose = verbose

    def __call__(self, module, inputs, output):
        # ← THIS runs during model(X), not during register_forward_hook()
        X_into_layer = inputs[0].detach().cpu()
        self.storage[self.layer_name] = X_into_layer
        if self.verbose:
            print(f"    HOOK RUNS → captured['{self.layer_name}'] = input shape {tuple(X_into_layer.shape)}")

print("LayerInputRecorder OK")


## `capture_inputs()` — attach `LayerInputRecorder` to every `nn.Linear`

**Why required:** GPTQ needs $X_l$ (activation **right side**, 3×3) per layer → $H = X_l^\top X_l$.

**Three steps:** ① register hooks → ② `model(X)` → ③ `handle.remove()`


In [ ]:
def capture_inputs(model, X, verbose=False):
    captured = {}
    hook_handles = []

    if verbose:
        print("① SETUP — register hooks (hook body does NOT run yet)\n")

    for layer_name, layer in model.named_modules():
        if not isinstance(layer, nn.Linear):
            continue

        recorder = LayerInputRecorder(layer_name, captured, verbose=verbose)
        handle = layer.register_forward_hook(recorder)
        hook_handles.append(handle)

        if verbose:
            print(f"    registered hook on '{layer_name}'")

    if verbose:
        print(f"\n    captured = {captured}  ← still empty!\n")
        print("② FORWARD — model(X)  (hooks run inside this line)\n")

    model.eval()
    with torch.no_grad():
        model(X)

    if verbose:
        print(f"\n    captured keys = {list(captured.keys())}\n")
        print("③ CLEANUP — remove hooks\n")

    for handle in hook_handles:
        handle.remove()

    return captured

print("capture_inputs OK")


### BEFORE vs AFTER `capture_inputs(ref, X)`

| | BEFORE | AFTER |
|--|--------|-------|
| **fc1** | only know `X` | `captures['fc1']` = `X` |
| **fc2** | don't know $h_1$ yet | `captures['fc2']` = $h_1$ |
| **fc3** | don't know $h_2$ yet | `captures['fc3']` = $h_2$ |

Hooks copy the **activation (right side)** entering each layer.


In [ ]:
# ── BEFORE capture_inputs ─────────────────────────────────────
# Only calibration X is known. No per-layer inputs yet.

captures = {}  # empty — nothing stored

print("BEFORE — captures dict:")
print(f"  captures = {captures}  (empty)")
print(f"  We only have X, shape {tuple(X.shape)}:\n")
show_matrix(X, "X  — given calibration (3 samples × 3 features)")

with torch.no_grad():
    h1_manual = F.relu(ref.fc1(X))
    h2_manual = F.relu(ref.fc2(h1_manual))

print("We can compute h₁, h₂ manually — but GPTQ pipeline uses hooks")
print("so the same numbers land in captures['fc2'] and captures['fc3'] automatically.")

In [ ]:
print("--- Run capture_inputs(verbose=True) ---\n")
captures = capture_inputs(ref, X, verbose=True)

print("AFTER — stored 3×3 matrices:")
for name, matrix in captures.items():
    print(f"  captures['{name}']  shape {tuple(matrix.shape)}")


### VISUAL — Step 3: BEFORE vs AFTER `capture_inputs` (3×3 proof)

Left column = what we knew **before**. Right = what each hook **stored**.

In [ ]:
# fc1 should store exactly X
diff_fc1 = (X.cpu() - captures["fc1"]).abs().max().item()
# fc2 should store h₁, fc3 should store h₂ (computed manually in BEFORE cell)
diff_fc2 = (h1.cpu() - captures["fc2"]).abs().max().item()
diff_fc3 = (h2.cpu() - captures["fc3"]).abs().max().item()

print("Proof — stored matrix matches expected input:")
print(f"  fc1: max|X − captures['fc1']|        = {diff_fc1:.2e}  (should be 0)")
print(f"  fc2: max|h₁ − captures['fc2']|       = {diff_fc2:.2e}  (should be 0)")
print(f"  fc3: max|h₂ − captures['fc3']|       = {diff_fc3:.2e}  (should be 0)")

show_row(
    [X, captures["fc1"], X.cpu() - captures["fc1"]],
    ["BEFORE: X", "AFTER: captures['fc1']", "difference"],
    suptitle="fc1 — hook stores the same 3×3 as calibration X",
)

show_row(
    [h1, captures["fc2"], h1.cpu() - captures["fc2"]],
    ["BEFORE: h₁ = ReLU(XW₁ᵀ)", "AFTER: captures['fc2']", "difference"],
    suptitle="fc2 — hook stores h₁ (ReLU output from fc1)",
)

show_row(
    [h2, captures["fc3"], h2.cpu() - captures["fc3"]],
    ["BEFORE: h₂ = ReLU(h₁W₂ᵀ)", "AFTER: captures['fc3']", "difference"],
    suptitle="fc3 — hook stores h₂ (ReLU output from fc2)",
)

show_row(
    [captures["fc1"], captures["fc2"], captures["fc3"]],
    ["captures['fc1']", "captures['fc2']", "captures['fc3']"],
    suptitle="Full store — these 3 matrices go into gptq_quantize() for H = Xₗᵀ Xₗ",
)

### VISUAL — Step 4: Hessian $H = X^\top X$ for **fc1** (3×3)

Diagonal = how much each input feature varies. Off-diagonal = co-variance.

In [ ]:
X_fc1 = captures["fc1"].float()
H_fc1 = X_fc1.t() @ X_fc1
show_row([X_fc1, H_fc1],
         ["X_fc1", "H = X_fc1ᵀ · X_fc1"],
         suptitle="Step 4 — build Hessian (3×3) from calibration data")

### VISUAL — Step 5: RTN on **fc1** — $W$ vs $\hat W$ vs error

In [ ]:
layer_fc1 = ref.fc1
W1 = layer_fc1.weight.data.clone()
st_rtn = rtn_quantize(layer_fc1)
sq = SymmetricQuantizer()
W1_rtn = sq.dequantize(st_rtn.q, st_rtn.s)

ql_rtn = QuantLinear(3, 3, st_rtn).to(DEVICE)
with torch.no_grad():
    y_rtn_fc1 = ql_rtn(X_fc1.to(DEVICE))
    y_fp_fc1 = F.linear(X_fc1.to(DEVICE), W1)

show_row([W1, W1_rtn, W1 - W1_rtn],
         ["fp16 W₁", "RTN Ŵ₁", "error"],
         suptitle="Step 5 — naive RTN on fc1 (no Hessian)")
show_row([y_fp_fc1, y_rtn_fc1, y_fp_fc1 - y_rtn_fc1],
         ["y = X W₁ᵀ", "ŷ RTN", "y − ŷ"],
         suptitle="RTN output error on calibration X")

### VISUAL — Step 6: GPTQ on **fc1** — column-by-column error propagation

After quantizing column $c$, remaining columns of $W$ are adjusted using $H^{-1}$.

In [ ]:
st_gptq, W0, H, Hinv, Q_gptq, snaps = gptq_quantize(
    ref.fc1, X_fc1, verbose=True)
sq = SymmetricQuantizer()
W1_gptq = sq.dequantize(st_gptq.q, st_gptq.s)

show_matrix(H, "H (after damping)")
show_matrix(Hinv, "H⁻¹ factor (used to spread error)")

fig, axes = plt.subplots(1, len(snaps), figsize=(3.5 * len(snaps), 3.2))
vmax = max(np.abs(W0.numpy()).max(), 1e-6)
for ax, (title, Wsnap) in zip(axes, snaps):
    A = Wsnap.numpy()
    im = ax.imshow(A, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=9)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{A[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.suptitle("Step 6 — W during GPTQ (watch cols 0→1→2 get quantized, rest adjusted)", y=1.05)
plt.tight_layout()
plt.show()

show_row([W0, W1_gptq, W0 - W1_gptq],
         ["fp16 W₁", "GPTQ Ŵ₁", "error"],
         suptitle="Final GPTQ weights vs fp16")
print("Integer Q from GPTQ:\n", Q_gptq.numpy())

### VISUAL — Step 7: RTN vs GPTQ output on fc1 (same int4, different Ŵ)

In [ ]:
ql_gptq = QuantLinear(3, 3, st_gptq).to(DEVICE)
with torch.no_grad():
    y_gptq_fc1 = ql_gptq(X_fc1.to(DEVICE))

mse_rtn = (y_fp_fc1 - y_rtn_fc1).pow(2).mean().item()
mse_gptq_l1 = (y_fp_fc1 - y_gptq_fc1).pow(2).mean().item()

show_row([y_fp_fc1, y_rtn_fc1, y_gptq_fc1],
         ["y fp16", "y RTN", "y GPTQ"],
         suptitle="fc1 outputs on same X_fc1")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["RTN", "GPTQ"], [mse_rtn, mse_gptq_l1], color=["#e74c3c", "#27ae60"])
ax.set_ylabel("output MSE vs fp16")
ax.set_title("fc1: lower bar = better")
plt.tight_layout()
plt.show()
print(f"fc1 output MSE — RTN: {mse_rtn:.4f}   GPTQ: {mse_gptq_l1:.4f}")

## Run — full 3-layer network: naive vs GPTQ

In [ ]:
# Phase A: naive int4 all layers
naive = Toy3Layer().to(DEVICE).eval()
naive.load_state_dict(ref.state_dict())
for name, layer in naive.named_modules():
    if isinstance(layer, nn.Linear):
        st = rtn_quantize(layer)
        replace_layer(naive, name, QuantLinear(3, 3, st).to(DEVICE))
with torch.no_grad():
    y_naive = naive(X)
mse_naive = (y_ref - y_naive).pow(2).mean().item()

# Phase D: GPTQ all layers
gptq_model = Toy3Layer().to(DEVICE).eval()
gptq_model.load_state_dict(ref.state_dict())
for name, layer in list(gptq_model.named_modules()):
    if isinstance(layer, nn.Linear):
        st = gptq_quantize(layer, captures[name].to(DEVICE))
        replace_layer(gptq_model, name, QuantLinear(3, 3, st).to(DEVICE))
with torch.no_grad():
    y_gptq = gptq_model(X)
mse_gptq = (y_ref - y_gptq).pow(2).mean().item()

print(f"End-to-end MSE — naive: {mse_naive:.4f}   GPTQ: {mse_gptq:.4f}")

### VISUAL — Step 8: final scorecard (all 3 layers quantized)

In [ ]:
show_row([y_ref, y_naive, y_gptq, y_ref - y_gptq],
         ["y fp16", "y naive", "y GPTQ", "y − y_GPTQ"],
         suptitle="Full network outputs (3×3)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["fp16 (0)", "naive int4", "GPTQ"], [0, mse_naive, mse_gptq],
       color=["#3498db", "#e74c3c", "#27ae60"])
ax.set_ylabel("end-to-end output MSE")
ax.set_title("3 layers × 3×3 weights — lower is better")
plt.tight_layout()
plt.show()

if mse_gptq < mse_naive:
    print(f"✓ GPTQ beats naive by {(1 - mse_gptq/mse_naive)*100:.1f}%")

## Recap — what each visual showed

| Step | Matrix | Meaning |
|------|--------|--------|
| 1 | $W_1, W_2, W_3$ | fp16 weights (left in Step 2) |
| 2 | $X$, $h_1$, $h_2$ | activations (right in Step 2) |
| 3 | `captures['fc1'..'fc3']` | `LayerInputRecorder` hook saved each $X_l$ |
| 4 | $H = X^\top X$ | Hessian from captured activation |
| 5 | $W$ vs $\hat W_{RTN}$ | naive int4 |
| 6 | $W$ after each column | GPTQ error spread |
| 7–8 | output MSE | GPTQ should beat RTN |

**Hook pattern everywhere:** `recorder = LayerInputRecorder(name, dict)` → `register_forward_hook(recorder)` → `model(X)` → `recorder.__call__` fills dict.

**Next:** `02_ocr_pipeline_quant_A.ipynb` — same hooks, Florence-2 + OCR image.
